# Federated Learning Benchmark Analysis

Анализ результатов сравнения стратегий агрегации на гетерогенных данных.

**Стратегии:** FedAvg, FedProx, FedAvgM, FedAdam, FedYogi, FedAdagrad, FedMedian, FedTrimmedAvg, Krum, MultiKrum, Bulyan

**Схемы разбиения:** IID, Dirichlet α=1.0 / 0.3 / 0.1

## 0. Imports

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Data Loading

Рекурсивно обходим `runs/`, ищем `*__rounds.csv`, `*__summary.json`, `*__clients.csv`.

Из имени папки извлекаем: dataset, scheme, alpha, strategy.

In [ ]:
RUNS_DIR = Path("../runs")


def parse_exp_name(name: str) -> dict:
    """Parse experiment folder name: dataset__scheme[__aALPHA]__strategy.
    
    Examples:
        cifar10__iid__fedavg
        mnist__dirichlet__a0p3__fedprox
    """
    parts = name.split("__")
    result = {"dataset": parts[0] if len(parts) > 0 else "unknown",
              "scheme": "iid", "alpha": None, "strategy": "unknown"}
    if len(parts) < 2:
        return result
    result["scheme"] = parts[1]
    idx = 2
    if idx < len(parts) and parts[idx].startswith("a") and re.match(r"a[0-9]+p?[0-9]*", parts[idx]):
        alpha_str = parts[idx][1:].replace("p", ".")
        try:
            result["alpha"] = float(alpha_str)
        except ValueError:
            pass
        idx += 1
    if idx < len(parts):
        result["strategy"] = parts[idx]
    return result


def load_experiment(exp_dir: Path) -> dict | None:
    """Load all data for one experiment directory."""
    rounds_files = list(exp_dir.glob("*__rounds.csv"))
    summary_files = list(exp_dir.glob("*__summary.json"))
    clients_files = list(exp_dir.glob("*__clients.csv"))

    if not rounds_files:
        return None

    rounds_df = pd.read_csv(rounds_files[0], skipinitialspace=True)
    summary = json.loads(summary_files[0].read_text()) if summary_files else {}
    clients_df = pd.read_csv(clients_files[0], skipinitialspace=True) if clients_files else pd.DataFrame()

    meta = parse_exp_name(exp_dir.name)
    return {
        "exp_name": exp_dir.name,
        "rounds": rounds_df,
        "clients": clients_df,
        "summary": summary,
        **meta,
    }


experiments = []
if RUNS_DIR.exists():
    for exp_dir in sorted(RUNS_DIR.iterdir()):
        if exp_dir.is_dir():
            exp = load_experiment(exp_dir)
            if exp is not None:
                experiments.append(exp)

print(f"Loaded {len(experiments)} experiments")
for e in experiments:
    print(f"  {e['exp_name']}: {len(e['rounds'])} rounds, strategy={e['strategy']}, "
          f"dataset={e['dataset']}, scheme={e['scheme']}, alpha={e['alpha']}")

In [ ]:
# Build a flat summary DataFrame
rows = []
for e in experiments:
    r = e["rounds"]
    s = e["summary"]
    scheme_label = e["scheme"]
    if e["alpha"] is not None:
        scheme_label += f" α={e['alpha']}"
    rows.append({
        "exp_name": e["exp_name"],
        "dataset": e["dataset"],
        "scheme": scheme_label,
        "strategy": e["strategy"],
        "final_acc": float(r["test_acc"].iloc[-1]) if len(r) > 0 else None,
        "best_acc": s.get("best_test_acc"),
        "best_round": s.get("best_round"),
        "rounds_to_80pct": s.get("rounds_to_80pct"),
        "rounds_to_90pct": s.get("rounds_to_90pct"),
        "rounds_to_95pct": s.get("rounds_to_95pct"),
        "total_comm_mb": s.get("total_comm_mb"),
        "total_wall_time_sec": s.get("total_wall_time_sec"),
        "final_std_train_loss": s.get("final_std_train_loss"),
    })

summary_df = pd.DataFrame(rows)
summary_df

## 2. Сводная таблица точности

`test_acc @ final round`: строки = стратегии, столбцы = схемы разбиения.

In [ ]:
for dataset in summary_df["dataset"].unique():
    sub = summary_df[summary_df["dataset"] == dataset]
    if sub.empty:
        continue
    pivot = sub.pivot_table(
        index="strategy", columns="scheme", values="final_acc", aggfunc="mean"
    )
    pivot = pivot.sort_index()
    print(f"\n=== {dataset.upper()} — Final test accuracy ===")
    display(pivot.style.format("{:.4f}").highlight_max(axis=0, color="#c8f7c5"))

## 3. Кривые сходимости

`test_acc` по раундам для каждой стратегии, сгруппировано по схеме разбиения.

In [ ]:
def plot_convergence(experiments, dataset_filter: str, title_suffix: str = ""):
    exps = [e for e in experiments if e["dataset"] == dataset_filter]
    if not exps:
        print(f"No experiments for dataset={dataset_filter}")
        return

    schemes = sorted({e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "") for e in exps})
    ncols = min(len(schemes), 3)
    nrows = (len(schemes) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

    for i, scheme in enumerate(schemes):
        ax = axes[i // ncols][i % ncols]
        for e in exps:
            e_scheme = e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "")
            if e_scheme != scheme:
                continue
            r = e["rounds"]
            if "round" in r.columns and "test_acc" in r.columns:
                ax.plot(r["round"], r["test_acc"], marker="o", markersize=3, label=e["strategy"])
        ax.set_title(f"{dataset_filter.upper()} | {scheme}")
        ax.set_xlabel("Round")
        ax.set_ylabel("Test Accuracy")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, alpha=0.3)

    for j in range(len(schemes), nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    fig.suptitle(f"Convergence curves — {dataset_filter.upper()} {title_suffix}", fontsize=13)
    plt.tight_layout()
    plt.show()


plot_convergence(experiments, "mnist")
plot_convergence(experiments, "cifar10")

## 4. Коммуникационная эффективность

`test_acc vs cum_comm_mb` — насколько точность растёт на МБ переданных данных.

In [ ]:
def plot_comm_efficiency(experiments, dataset_filter: str):
    exps = [e for e in experiments if e["dataset"] == dataset_filter]
    if not exps:
        print(f"No experiments for dataset={dataset_filter}")
        return

    schemes = sorted({e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "") for e in exps})
    ncols = min(len(schemes), 3)
    nrows = (len(schemes) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

    for i, scheme in enumerate(schemes):
        ax = axes[i // ncols][i % ncols]
        for e in exps:
            e_scheme = e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "")
            if e_scheme != scheme:
                continue
            r = e["rounds"]
            if "cum_comm_mb" in r.columns and "test_acc" in r.columns:
                ax.plot(r["cum_comm_mb"], r["test_acc"], marker="o", markersize=3, label=e["strategy"])
        ax.set_title(f"{dataset_filter.upper()} | {scheme}")
        ax.set_xlabel("Cumulative Communication (MB)")
        ax.set_ylabel("Test Accuracy")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, alpha=0.3)

    for j in range(len(schemes), nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    fig.suptitle(f"Communication efficiency — {dataset_filter.upper()}", fontsize=13)
    plt.tight_layout()
    plt.show()


plot_comm_efficiency(experiments, "mnist")
plot_comm_efficiency(experiments, "cifar10")

## 5. Время раундов

Boxplot `round_wall_time_sec` по стратегиям.

In [ ]:
def plot_round_times(experiments, dataset_filter: str):
    exps = [e for e in experiments if e["dataset"] == dataset_filter]
    if not exps:
        print(f"No experiments for dataset={dataset_filter}")
        return

    data_by_strategy: dict = {}
    for e in exps:
        r = e["rounds"]
        if "round_wall_time_sec" not in r.columns:
            continue
        strat = e["strategy"]
        times = r["round_wall_time_sec"].dropna().tolist()
        data_by_strategy.setdefault(strat, []).extend(times)

    if not data_by_strategy:
        print("No round_wall_time_sec data found")
        return

    strategies = sorted(data_by_strategy.keys())
    fig, ax = plt.subplots(figsize=(max(8, len(strategies) * 1.2), 5))
    ax.boxplot([data_by_strategy[s] for s in strategies], labels=strategies, patch_artist=True)
    ax.set_title(f"Round Wall Time — {dataset_filter.upper()}")
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Round wall time (sec)")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_round_times(experiments, "mnist")
plot_round_times(experiments, "cifar10")

## 6. Fairness

`std_train_loss` по раундам для каждой стратегии — чем ниже, тем справедливее распределение качества между клиентами.

In [ ]:
def plot_fairness(experiments, dataset_filter: str):
    exps = [e for e in experiments if e["dataset"] == dataset_filter]
    if not exps:
        print(f"No experiments for dataset={dataset_filter}")
        return

    schemes = sorted({e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "") for e in exps})
    ncols = min(len(schemes), 3)
    nrows = (len(schemes) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

    for i, scheme in enumerate(schemes):
        ax = axes[i // ncols][i % ncols]
        for e in exps:
            e_scheme = e["scheme"] + (f" α={e['alpha']}" if e["alpha"] is not None else "")
            if e_scheme != scheme:
                continue
            r = e["rounds"]
            if "round" in r.columns and "std_train_loss" in r.columns:
                ax.plot(r["round"], r["std_train_loss"], marker="o", markersize=3, label=e["strategy"])
        ax.set_title(f"{dataset_filter.upper()} | {scheme}")
        ax.set_xlabel("Round")
        ax.set_ylabel("Std Train Loss (Fairness proxy)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for j in range(len(schemes), nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    fig.suptitle(f"Fairness (std_train_loss) — {dataset_filter.upper()}", fontsize=13)
    plt.tight_layout()
    plt.show()


plot_fairness(experiments, "mnist")
plot_fairness(experiments, "cifar10")

## 7. Итоговая сводная таблица с Fairness

In [ ]:
cols_show = ["dataset", "scheme", "strategy", "final_acc", "best_acc", "best_round",
             "rounds_to_80pct", "rounds_to_90pct", "rounds_to_95pct",
             "total_comm_mb", "total_wall_time_sec", "final_std_train_loss"]
existing_cols = [c for c in cols_show if c in summary_df.columns]
display(
    summary_df[existing_cols]
    .sort_values(["dataset", "scheme", "final_acc"], ascending=[True, True, False])
    .style.format({
        "final_acc": "{:.4f}",
        "best_acc": "{:.4f}",
        "total_comm_mb": "{:.1f}",
        "total_wall_time_sec": "{:.1f}",
        "final_std_train_loss": "{:.4f}",
    }, na_rep="—")
    .highlight_max(subset=["final_acc", "best_acc"], color="#c8f7c5")
    .highlight_min(subset=["final_std_train_loss", "total_comm_mb"], color="#c8f7c5")
)